In [31]:
import numpy as np, pandas as pd


index = np.repeat(pd.date_range(start='2025-01-01',end='2025-01-31', freq='1d',inclusive='left'), repeats=24) 

A = np.random.normal(loc=0, scale=1, size=index.shape[0])
B = 2 * A + np.random.normal(loc=0, scale=1, size=len(A))


dataA = pd.DataFrame(data={"A":A, "B":B},index=index)

dataA

,A,B
2025-01-01,1.743853,4.071002
2025-01-01,0.375478,1.047802
2025-01-01,-0.164655,0.432173
2025-01-01,0.465811,1.558166
2025-01-01,-0.014944,1.057744
...,...,...
2025-01-30,-1.195005,-2.358996
2025-01-30,0.195303,-0.469044
2025-01-30,-1.929525,-4.158636
2025-01-30,0.338404,0.091614


In [32]:
dataA_agg = (
    dataA.groupby(
        dataA.index
    )
    .agg(["sum","max"])
    # .reset_index()
)



C1 = 3* dataA_agg[('B','sum')] + np.random.normal(loc=0, scale=1, size=len(dataA_agg[('B','sum')]))
C2 =  dataA_agg[('B','max')]


D = 4 * C1 + np.random.normal(loc=0, scale=1, size=len(C1))
E = np.cumsum(C2)
dataB = pd.DataFrame(data=dict(C=C1, D=D, E=E))
dataB

,C,D,E
2025-01-01,6.257291,25.206815,4.071002
2025-01-02,-12.834589,-51.877916,7.816545
2025-01-03,-12.397990,-47.960374,12.696889
2025-01-04,20.857721,83.478432,18.337037
2025-01-05,-14.075583,-57.614490,22.050836
2025-01-06,-7.331465,-28.513764,26.589074
2025-01-07,-5.168454,-18.988071,30.723251
2025-01-08,27.646534,110.551547,36.016850
2025-01-09,-29.075782,-114.824833,39.721263
2025-01-10,31.540344,125.792356,45.397916


In [33]:
dataA =dataA.reset_index(names="!index")
dataB =dataB.reset_index(names="!index")

In [34]:
import networkx as nx
causal_graphA = nx.DiGraph([('A', 'B')])
causal_graphB = nx.DiGraph([('C', 'D'),('D','E')])

In [35]:
from dowhy import gcm

causal_modelA = gcm.StructuralCausalModel(causal_graphA)
causal_modelB = gcm.StructuralCausalModel(causal_graphB)

In [36]:
from dowhy.gcm.defined_causal_mechanisms import DefinedConditionalStochasticModel


causal_modelB.set_causal_mechanism("E",DefinedConditionalStochasticModel(lambda x: np.cumsum(x)))

In [ ]:
auto_assignment_summaryA = gcm.auto.assign_causal_mechanisms(causal_modelA, based_on=dataA,experimental_allow_nans=True)
auto_assignment_summaryB = gcm.auto.assign_causal_mechanisms(causal_modelB, based_on=dataB,experimental_allow_nans=True)

Optionally, we can get more insights from the auto assignment process:

In the real world, the data comes as an opaque stream of values, where we typically don't know how one
variable influences another. The graphical causal models can help us to deconstruct these causal
relationships again, even though we didn't know them before.

## Step 2: Fitting the SCM to the data

With the data at hand and the graph constructed earlier, we can now train the SCM using `fit`:

In [ ]:
gcm.fit(causal_modelA, dataA)
gcm.fit(causal_modelB, dataB)

Fitting means, we learn the generative models of the variables in the SCM according to the data.

Once fitted, we can also obtain more insights into the model performances:

In [ ]:
print(gcm.evaluate_causal_model(causal_modelA, dataA))

In [ ]:
print(gcm.evaluate_causal_model(causal_modelB, dataB))

In [ ]:

links = {
    ("B", "C","!index"):"sum"
    ,
    ("B", "C2","!index"):"max"
    
}

In [ ]:
causal_model_composite = gcm.StructuralCausalModelComposite(
    [causal_modelA, causal_modelB], links=links
)

## Step 3: Answering a causal query based on the SCM

The last step, answering a causal question, is our actual goal. E.g. we could ask the question "What will happen to the variable Z if I intervene on Y?".

This can be done via the `interventional_samples` function. Here’s how:

In [ ]:
samples = gcm.interventional_samples(causal_model_composite,
                                     {'B': lambda B: B },
                                    observed_data= dataA[["!index","A"]])

[pd.DataFrame({k:v for k,v in samples.items() if len(v)== length}) for length in set([ len(sv) for sv  in samples.values()])]